In [9]:
# Import required libraries
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import StandardScaler
from torch.nn.utils.rnn import pad_sequence
import json

# Load the final processed datasets
train_df = pd.read_parquet("../data/processed/train_final.parquet")
val_df = pd.read_parquet("../data/processed/val_final.parquet")

print(train_df.shape, val_df.shape)

(1085255, 73) (230301, 73)


In [3]:
# Feature columns (same as before)
exclude_cols = ["patient_id", "SepsisLabel"]
feature_cols = [c for c in train_df.columns if c not in exclude_cols]

# Function to convert dataframe into per-patient sequences
def create_sequences(data, feature_cols, max_len=100):
    sequences = []
    labels = []
    for pid in data["patient_id"].unique():
        patient_data = data[data["patient_id"] == pid].sort_values("ICULOS")
        seq = patient_data[feature_cols].values
        label = patient_data["SepsisLabel"].values
        if len(seq) > max_len:
            seq = seq[:max_len]
            label = label[:max_len]
        sequences.append(seq)
        labels.append(label)
    return sequences, labels

# Function to pad sequences into equal-length tensors
def prepare_padded_data(sequences, labels):
    seq_tensors = [torch.tensor(seq, dtype=torch.float32) for seq in sequences]
    label_tensors = [torch.tensor(lab, dtype=torch.float32) for lab in labels]
    lengths = [len(seq) for seq in sequences]
    padded_seqs = pad_sequence(seq_tensors, batch_first=True, padding_value=0)
    padded_labels = pad_sequence(label_tensors, batch_first=True, padding_value=0)
    return padded_seqs, padded_labels, lengths

print("functions ready")

functions ready


In [4]:
# Create sequences for train and validation sets
train_sequences, train_labels = create_sequences(train_df, feature_cols)
val_sequences, val_labels = create_sequences(val_df, feature_cols)

print("Sequences created:", len(train_sequences), len(val_sequences))

# Pad the sequences
train_padded_seqs, train_padded_labels, train_lengths = prepare_padded_data(train_sequences, train_labels)
val_padded_seqs, val_padded_labels, val_lengths = prepare_padded_data(val_sequences, val_labels)

print("Padded shapes:", train_padded_seqs.shape, val_padded_seqs.shape)

# Normalize features using StandardScaler (fit on train only)
scaler = StandardScaler()
n_patients_train, max_len, n_feat = train_padded_seqs.shape
train_flat = train_padded_seqs.numpy().reshape(-1, n_feat)
scaler.fit(train_flat)

train_scaled = scaler.transform(train_flat).reshape(n_patients_train, max_len, n_feat)
train_padded_seqs_scaled = torch.tensor(train_scaled, dtype=torch.float32)

n_patients_val = val_padded_seqs.shape[0]
val_flat = val_padded_seqs.numpy().reshape(-1, n_feat)
val_scaled = scaler.transform(val_flat).reshape(n_patients_val, max_len, n_feat)
val_padded_seqs_scaled = torch.tensor(val_scaled, dtype=torch.float32)

print("Normalization done")

Sequences created: 28235 6050
Padded shapes: torch.Size([28235, 100, 71]) torch.Size([6050, 100, 71])
Normalization done


In [5]:
# Define our GRU-based neural network
# GRU is similar to LSTM but simpler/faster (fewer internal calculations)
class SepsisGRU(nn.Module):
    def __init__(self, n_features, hidden_size=64):
        super().__init__()
        
        # GRU layer instead of LSTM - does similar job with a simpler internal structure
        self.gru = nn.GRU(input_size=n_features, hidden_size=hidden_size, 
                           batch_first=True)
        
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        gru_out, _ = self.gru(x)
        gru_out = self.dropout(gru_out)
        out = torch.sigmoid(self.fc(gru_out))
        return out.squeeze(-1)

# Create the GRU model
gru_model = SepsisGRU(n_features=71, hidden_size=64)
print(gru_model)

SepsisGRU(
  (gru): GRU(71, 64, batch_first=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)


In [6]:
# Calculate positive class weight using real (non-padded) labels
all_real_labels = np.concatenate(train_labels)
real_neg = (all_real_labels == 0).sum()
real_pos = (all_real_labels == 1).sum()
pos_weight_value = real_neg / real_pos

print("Positive class weight:", pos_weight_value)

# Loss function and optimizer
criterion = nn.BCELoss(reduction='none')
optimizer_gru = torch.optim.Adam(gru_model.parameters(), lr=0.001)

# Training loop (same structure as LSTM, 15 epochs which worked best)
n_epochs = 15
batch_size = 64
n_train = train_padded_seqs_scaled.shape[0]
train_lengths_tensor = torch.tensor(train_lengths)

for epoch in range(n_epochs):
    gru_model.train()
    total_loss = 0
    
    for i in range(0, n_train, batch_size):
        batch_seqs = train_padded_seqs_scaled[i:i+batch_size]
        batch_labels = train_padded_labels[i:i+batch_size]
        batch_lengths = train_lengths_tensor[i:i+batch_size]
        
        max_len_batch = batch_seqs.shape[1]
        mask = torch.arange(max_len_batch).unsqueeze(0) < batch_lengths.unsqueeze(1)
        mask = mask.float()
        
        optimizer_gru.zero_grad()
        predictions = gru_model(batch_seqs)
        raw_loss = criterion(predictions, batch_labels)
        masked_loss = (raw_loss * mask).sum() / mask.sum()
        masked_loss.backward()
        optimizer_gru.step()
        
        total_loss += masked_loss.item()
    
    avg_loss = total_loss / (n_train / batch_size)
    print(f"Epoch {epoch+1}/{n_epochs}, Average Loss: {avg_loss:.4f}")

Positive class weight: 64.21048730776357
Epoch 1/15, Average Loss: 0.0896
Epoch 2/15, Average Loss: 0.0740
Epoch 3/15, Average Loss: 0.0723
Epoch 4/15, Average Loss: 0.0715
Epoch 5/15, Average Loss: 0.0709
Epoch 6/15, Average Loss: 0.0704
Epoch 7/15, Average Loss: 0.0699
Epoch 8/15, Average Loss: 0.0697
Epoch 9/15, Average Loss: 0.0693
Epoch 10/15, Average Loss: 0.0688
Epoch 11/15, Average Loss: 0.0685
Epoch 12/15, Average Loss: 0.0680
Epoch 13/15, Average Loss: 0.0676
Epoch 14/15, Average Loss: 0.0672
Epoch 15/15, Average Loss: 0.0665


In [7]:
gru_model.eval()

with torch.no_grad():
    val_predictions_gru = gru_model(val_padded_seqs_scaled)

all_val_preds_gru = []
all_val_labels_gru = []

for i in range(len(val_lengths)):
    real_len = val_lengths[i]
    all_val_preds_gru.extend(val_predictions_gru[i, :real_len].numpy())
    all_val_labels_gru.extend(val_padded_labels[i, :real_len].numpy())

all_val_preds_gru = np.array(all_val_preds_gru)
all_val_labels_gru = np.array(all_val_labels_gru)

auroc_gru = roc_auc_score(all_val_labels_gru, all_val_preds_gru)
auprc_gru = average_precision_score(all_val_labels_gru, all_val_preds_gru)

print("GRU AUROC:", auroc_gru)
print("GRU AUPRC:", auprc_gru)

GRU AUROC: 0.7777293387657541
GRU AUPRC: 0.07657722481405074


In [10]:
torch.save(gru_model.state_dict(), "../models/gru_model.pt")

gru_results = {
    "model": "GRU",
    "AUROC": auroc_gru,
    "AUPRC": auprc_gru,
    "epochs": 15
}
with open("../reports/gru_results.json", "w") as f:
    json.dump(gru_results, f, indent=2)

print("GRU saved!")

GRU saved!


In [11]:
# TCN uses 1D convolutions with "dilation" - this lets it see far-back patterns efficiently
class SepsisTCN(nn.Module):
    def __init__(self, n_features, num_channels=64):
        super().__init__()
        
        # First convolutional layer - looks at nearby time-steps
        self.conv1 = nn.Conv1d(n_features, num_channels, kernel_size=3, 
                                padding=2, dilation=1)
        
        # Second convolutional layer - dilation=2 means it looks further back in time
        self.conv2 = nn.Conv1d(num_channels, num_channels, kernel_size=3, 
                                padding=4, dilation=2)
        
        # Third convolutional layer - dilation=4 means it looks even further back
        self.conv3 = nn.Conv1d(num_channels, num_channels, kernel_size=3, 
                                padding=8, dilation=4)
        
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(num_channels, 1)
    
    def forward(self, x):
        # Conv1d expects shape (batch, features, time) but our data is (batch, time, features)
        # So we need to swap dimensions first
        x = x.permute(0, 2, 1)
        
        # Pass through the three convolutional layers
        out = self.relu(self.conv1(x))
        out = out[:, :, :x.shape[2]]  # trim extra padding to match original length
        
        out = self.relu(self.conv2(out))
        out = out[:, :, :x.shape[2]]
        
        out = self.relu(self.conv3(out))
        out = out[:, :, :x.shape[2]]
        
        out = self.dropout(out)
        
        # Swap dimensions back to (batch, time, features) for the final layer
        out = out.permute(0, 2, 1)
        out = torch.sigmoid(self.fc(out))
        
        return out.squeeze(-1)

# Create the TCN model
tcn_model = SepsisTCN(n_features=71, num_channels=64)
print(tcn_model)

SepsisTCN(
  (conv1): Conv1d(71, 64, kernel_size=(3,), stride=(1,), padding=(2,))
  (conv2): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(4,), dilation=(2,))
  (conv3): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(8,), dilation=(4,))
  (relu): ReLU()
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)


In [16]:
# Loss and optimizer for TCN
optimizer_tcn = torch.optim.Adam(tcn_model.parameters(), lr=0.001)

n_epochs = 15
batch_size = 64
n_train = train_padded_seqs_scaled.shape[0]
train_lengths_tensor = torch.tensor(train_lengths)

for epoch in range(n_epochs):
    tcn_model.train()
    total_loss = 0
    
    for i in range(0, n_train, batch_size):
        batch_seqs = train_padded_seqs_scaled[i:i+batch_size]
        batch_labels = train_padded_labels[i:i+batch_size]
        batch_lengths = train_lengths_tensor[i:i+batch_size]
        
        max_len_batch = batch_seqs.shape[1]
        mask = torch.arange(max_len_batch).unsqueeze(0) < batch_lengths.unsqueeze(1)
        mask = mask.float()
        
        optimizer_tcn.zero_grad()
        predictions = tcn_model(batch_seqs)
        raw_loss = criterion(predictions, batch_labels)
        masked_loss = (raw_loss * mask).sum() / mask.sum()
        masked_loss.backward()
        optimizer_tcn.step()
        
        total_loss += masked_loss.item()
    
    avg_loss = total_loss / (n_train / batch_size)
    print(f"Epoch {epoch+1}/{n_epochs}, Average Loss: {avg_loss:.4f}")

Epoch 1/15, Average Loss: 0.0561
Epoch 2/15, Average Loss: 0.0549
Epoch 3/15, Average Loss: 0.0550
Epoch 4/15, Average Loss: 0.0542
Epoch 5/15, Average Loss: 0.0541
Epoch 6/15, Average Loss: 0.0537
Epoch 7/15, Average Loss: 0.0536
Epoch 8/15, Average Loss: 0.0531
Epoch 9/15, Average Loss: 0.0527
Epoch 10/15, Average Loss: 0.0523
Epoch 11/15, Average Loss: 0.0521
Epoch 12/15, Average Loss: 0.0517
Epoch 13/15, Average Loss: 0.0515
Epoch 14/15, Average Loss: 0.0512
Epoch 15/15, Average Loss: 0.0506


In [17]:
tcn_model.eval()

with torch.no_grad():
    val_predictions_tcn = tcn_model(val_padded_seqs_scaled)

all_val_preds_tcn = []
all_val_labels_tcn = []

for i in range(len(val_lengths)):
    real_len = val_lengths[i]
    all_val_preds_tcn.extend(val_predictions_tcn[i, :real_len].numpy())
    all_val_labels_tcn.extend(val_padded_labels[i, :real_len].numpy())

all_val_preds_tcn = np.array(all_val_preds_tcn)
all_val_labels_tcn = np.array(all_val_labels_tcn)

auroc_tcn = roc_auc_score(all_val_labels_tcn, all_val_preds_tcn)
auprc_tcn = average_precision_score(all_val_labels_tcn, all_val_preds_tcn)

print("TCN AUROC:", auroc_tcn)
print("TCN AUPRC:", auprc_tcn)

TCN AUROC: 0.6672686310144237
TCN AUPRC: 0.03868657871051331


In [18]:
# Save the TCN model
torch.save(tcn_model.state_dict(), "../models/tcn_model.pt")

# Save results
tcn_results = {
    "model": "TCN",
    "AUROC": auroc_tcn,
    "AUPRC": auprc_tcn,
    "epochs": 15
}
with open("../reports/tcn_results.json", "w") as f:
    json.dump(tcn_results, f, indent=2)

print("TCN saved!")

TCN saved!
